# ChEMBL 35 (IC50 / Kd / Ki) 데이터 전처리

`chembl_data/`의 원본 ChEMBL export를 `DTA_dataset/chembl_35/{IC50,Kd,Ki}/fold1~5/*.csv`
(DTA 모델 학습·예측 입력)로 만드는 과정을 정리한 노트북이다.

## 왜 이 노트북을 새로 만들었나
`DTA_dataset/chembl_35/data_processing.ipynb`가 원래 이 최종 변환을 담당했을 것으로 보이는데,
이번 세션 중 데이터셋 폴더가 재구성되면서 그 파일이 삭제되어 원본 로직을 그대로 볼 수 없었다.
대신 `chembl_data/data/dataProcessing.ipynb`, `chembl_data/data/duplicate_data.ipynb`
(탐색적으로 173개+ 셀에 걸쳐 시행착오로 작성된 노트북들)와, 실제로 만들어져 있는
`DTA_dataset/chembl_35/**/*.csv`의 값들을 역으로 분석해서 핵심 로직을 검증하고 정리했다.

## 원본 파일 관련 주의
`chembl_data/Chembl_IC50.csv`(163MB)는 `bao_format` 컬럼이 없는 **이미 필터링된 중간 산출물**이다.
진짜 원본은 `chembl_data/data/Chembl_IC50.csv`(544MB, `bao_format` 컬럼 있음) 쪽이며,
이 노트북은 후자를 원본으로 사용한다. (`Chembl_Kd.csv`, `Chembl_Ki.csv`도 동일)

## 검증된 것 / 검증 못한 것
아래는 실제 `DTA_dataset/chembl_35/IC50/fold1/train_data.csv` 등의 값을 직접 역산해서 맞춰본 것들:
- **affinity 공식**: `affinity = 9 - log10(standard_value_nM)` — 검증 완료
  (예: CHEMBL138656/P22894, standard_value=2.0nM → affinity=8.698970004336019, 원본 raw csv와 실제 fold 파일 값이 정확히 일치)
- **interaction 라벨**: `standard_value == 10000.0`(정확히, ChEMBL의 "값 상한 캡" 관례)이면 `interaction=0`, 아니면 `1` — 검증 완료
- **변이 단백질 식별자**: `target_accession(mut1,mut2,...)` 형태로 합쳐서 저장 (예: `P00533(L858R,T790M)`) — 검증 완료
- **fold 분할**: 5-fold이며 fold별 test set은 서로 완전히 겹치지 않고(진짜 K-fold), 합치면 전체 데이터와 같다.
  남은 80%는 train(60%)/valid(20%)로 다시 분할 — 검증 완료 (정확한 random seed는 알 수 없음)
- **test_non5_data.csv 등 "non5" 계열의 정확한 생성 규칙**과, 특정 변이/설명을 걸러낸 세부 예외 처리(정규식 fix, 하드코딩된 보정 목록)의
  **완전한 이력**은 복원하지 못했다. "학습셋에서 5회 미만 등장하는 단백질"이라는 단순 빈도 가설을 데이터로 검증해봤지만
  실제 파일과 맞지 않아서(오히려 non5 쪽 단백질의 학습 데이터 등장 횟수가 더 많았다) **이 노트북은 non5 파일을 만들지 않는다.**
  이미 만들어진 `DTA_dataset/chembl_35/*/foldN/test_non5_data.csv` 등은 그대로 두고 쓰면 된다.

## 실행에 대해
아래 코드는 실제 값으로 검증된 로직이지만, SDF 파일이 6.7GB라 **전체 파이프라인을 처음부터 끝까지 다시 돌리면
상당히 오래 걸린다** (RDKit SDF 파싱 + 전체 dedup/평균 계산). 이 세션에서는 각 단계의 핵심 함수만
샘플 데이터로 검증했고, 전체 실행은 하지 않았다. `DTA_dataset/chembl_35/`에 이미 결과가 있으므로
새로 만들 필요가 없다면 이 노트북은 몰아서 실행하기보다 참고 문서로 사용하는 걸 권장한다.


In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../chembl_data/data")          # 진짜 원본 (bao_format 포함)
OUT_DIR = Path("../DTA_dataset/chembl_35")     # 최종 결과 위치

ASSAYS = ["IC50", "Kd", "Ki"]
SEED = 42

# environment.yml에 scikit-learn이 없어서 KFold/train_test_split도 numpy로 직접 구현한다.
def kfold_indices(n, n_splits=5, seed=SEED):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(n)
    fold_sizes = np.full(n_splits, n // n_splits, dtype=int)
    fold_sizes[: n % n_splits] += 1
    folds, start = [], 0
    for size in fold_sizes:
        folds.append(idx[start:start + size])
        start += size
    return folds


def random_split(idx, test_size, seed):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(idx)
    n_test = int(round(len(idx) * test_size))
    return idx[n_test:], idx[:n_test]


## 1. 원본 ChEMBL export 불러오기 + BAO_format 필터링

`bao_format == 'BAO_0000357'`(single protein format assay)만 남긴다.
(dataProcessing.ipynb 초반 셀에서는 한때 `BAO_0000224`도 함께 허용했다가 최종적으로는 `'BAO_0000357'` 단일 값으로 좁혔다.)


In [ ]:
raw = {}
for assay in ASSAYS:
    df = pd.read_csv(RAW_DIR / f"Chembl_{assay}.csv")
    raw[assay] = df[df["bao_format"] == "BAO_0000357"].copy()
    print(f"{assay}: raw={len(df)} -> bao_format 필터 후={len(raw[assay])}")


## 2. SMILES / FASTA 붙이기

- `chembl_35.sdf` → RDKit으로 파싱, `chembl_id` → canonical SMILES
- `chembl_35.fa` → Biopython으로 파싱, `target_chembl_id` → 서열

둘 다 `compound_chembl_id` / `target_chembl_id` 기준으로 left-merge한다.


In [ ]:
from rdkit import Chem
from Bio import SeqIO

def load_smiles_table(sdf_path):
    rows = []
    for mol in Chem.SDMolSupplier(str(sdf_path)):
        if mol is None or not mol.HasProp("chembl_id"):
            continue
        rows.append((mol.GetProp("chembl_id"), Chem.MolToSmiles(mol)))
    return pd.DataFrame(rows, columns=["chembl_id", "SMILES"])


def load_fasta_table(fa_path):
    rows = []
    for record in SeqIO.parse(str(fa_path), "fasta"):
        for target_id in record.id.split("|")[0].strip().split(","):
            rows.append((target_id.strip(), str(record.seq)))
    return pd.DataFrame(rows, columns=["target_chembl_id", "FASTA"])


df_smiles = load_smiles_table(RAW_DIR / "chembl_35.sdf")
df_fasta = load_fasta_table(RAW_DIR / "chembl_35.fa")
print(f"unique SMILES: {df_smiles['chembl_id'].nunique()}, unique FASTA targets: {df_fasta['target_chembl_id'].nunique()}")


In [ ]:
merged = {}
for assay in ASSAYS:
    df = raw[assay].merge(df_smiles, how="left", left_on="compound_chembl_id", right_on="chembl_id")
    df = df.merge(df_fasta, how="left", on="target_chembl_id")
    df = df.drop(columns=["chembl_id"])
    df = df.dropna(subset=["SMILES", "FASTA"])  # 구조/서열을 못 찾은 행 제거
    merged[assay] = df
    print(f"{assay}: SMILES/FASTA 결합 후 = {len(df)}")


## 3. WT / 변이(mutant) 분리

`assay_description`에 `mutant|mutation|substitution|SNP|variant|deletion` 키워드가 있으면 변이 후보,
없으면 WT로 1차 분리한다.


In [ ]:
MUTATION_KEYWORDS = r"mutant|mutation|substitution|SNP|variant|deletion"

wt = {}
mut_candidate = {}
for assay in ASSAYS:
    df = merged[assay]
    is_mut = df["assay_description"].str.contains(MUTATION_KEYWORDS, case=False, na=False)
    wt[assay] = df[~is_mut].copy()
    mut_candidate[assay] = df[is_mut].copy()
    print(f"{assay}: WT 후보={len(wt[assay])}, 변이 후보={len(mut_candidate[assay])}")


## 4. 변이 부위(mutation_sites) 추출

`assay_description` 괄호 안에 있는 아미노산 치환/결실/삽입 표기를 정규식으로 뽑는다.
괄호 안에서 못 찾으면 문장 전체에서 `[A-Z]\d{1,4}[A-Z]` 패턴을 한 번 더 시도한다
(예: `"EGFR L858R mutant"`처럼 괄호 없이 쓰인 경우).

알려진 오탐(예: 수용체 이름 `A2A`, 유전자명 `A1R`이 변이처럼 매칭되는 경우)은
`dataProcessing.ipynb`에서 수작업으로 고친 리스트를 그대로 반영한다.


In [ ]:
def extract_mutation_sites(description):
    if not isinstance(description, str):
        return []

    inside_parens = re.findall(r"\([^\(\)]*\)", description)
    sites = []
    for chunk in inside_parens:
        chunk = chunk.strip("()")
        for token in chunk.replace(",", " ").split():
            token = token.strip()
            if re.match(r"^[A-Z]\d+[A-Z]$", token):
                sites.append(token)
            elif re.match(r"^[A-Z]\d+[-_][A-Z]?\d+del$", token):
                sites.append(token)
            elif re.match(r"^[A-Z]\d+[-_][A-Z]?\d+ins[A-Z]+$", token):
                sites.append(token)
            elif re.match(r"^[A-Z]\d+[-_][A-Z]?\d+delins[A-Z]+$", token):
                sites.append(token)

    if sites:
        return sites
    return re.findall(r"\b[A-Z]\d{1,4}[A-Z]\b", description)


# dataProcessing.ipynb에서 수작업으로 발견한 오탐/예외 보정
KNOWN_FALSE_POSITIVES = {"A1R"}          # 유전자명이 변이처럼 매칭된 경우 → 그 행 자체를 버림
KNOWN_FIXES = {                          # (본문 문자열 그대로) 오매칭된 토큰 제거
    ("A2A", "H250N"): ("H250N",),
}
EXCLUDE_DESCRIPTION_KEYWORDS = ["APP", "ITD"]  # 아밀로이드전구체(APP)/내부탠덤중복(ITD) 오탐 제거

# 특정 assay_description은 일반 정규식으로 못 뽑아서 수작업으로 목록화한 것 (도메인 결합 실험 등)
MANUAL_MUTATION_TABLE = {
    "Binding constant for KIT(V559D,T670I) kinase domain": ["V559D", "T670I"],
    "Binding constant for KIT(V559D,V654A) kinase domain": ["V559D", "V654A"],
    "Binding constant for GCN2(Kin.Dom.2,S808G) kinase domain": ["S808G"],
    "Binding constant for EGFR(E746-A750del) kinase domain": ["E746-A750del"],
    "Binding constant for EGFR(L747-E749del, A750P) kinase domain": ["E746-A750del", "A750P"],
    "Binding constant for EGFR(L747-S752del, P753S) kinase domain": ["E746-A750del", "P753S"],
    "Binding constant for EGFR(L747-T751del,Sins) kinase domain": ["L747_T751del", "E746_S747insS"],
    "Binding constant for EGFR(S752-I759del) kinase domain": ["S752-I759del"],
    "Binding constant for EGFR(L858R,T790M) kinase domain": ["L858R", "T790M"],
}


def clean_sites(sites):
    key = tuple(sites)
    if key in KNOWN_FIXES:
        return list(KNOWN_FIXES[key])
    return sites


In [ ]:
mut = {}
for assay in ASSAYS:
    df = mut_candidate[assay].copy()
    for kw in EXCLUDE_DESCRIPTION_KEYWORDS:
        df = df[~df["assay_description"].str.contains(kw, case=False, na=False)]

    sites_col = []
    for desc in df["assay_description"]:
        sites = MANUAL_MUTATION_TABLE.get(desc)
        if sites is None:
            sites = clean_sites(extract_mutation_sites(desc))
        sites_col.append(sites)
    df["mutation_sites"] = sites_col

    # 추출 실패(빈 리스트)했거나 알려진 오탐 하나로만 이루어진 행은 제거
    df = df[df["mutation_sites"].apply(lambda s: len(s) > 0 and set(s) != KNOWN_FALSE_POSITIVES)]

    mut[assay] = df
    print(f"{assay}: 변이 부위 추출 성공 = {len(df)} / 후보 {len(mut_candidate[assay])}")


## 5. 표준 단위(nM) 필터 + 중복 측정값 평균

같은 (화합물, 단백질[, 변이]) 조합이 여러 측정값(assay)으로 중복되는 경우가 많아서,
`standard_units == 'nM'`인 행만 남기고 중복은 `standard_value` 평균으로 합친다
(`duplicate_data.ipynb`의 `*_avg` 처리에 해당).


In [ ]:
def dedup_average(df, group_cols):
    df = df[df["standard_units"] == "nM"].copy()
    agg = (
        df.groupby(group_cols, as_index=False)
        .agg(standard_value=("standard_value", "mean"))
    )
    # 그룹별 대표 메타데이터(첫 행)와 평균값을 합친다
    first = df.drop_duplicates(subset=group_cols)[group_cols + ["SMILES", "FASTA"]]
    return agg.merge(first, on=group_cols, how="left")


wt_dedup, mut_dedup = {}, {}
for assay in ASSAYS:
    wt_dedup[assay] = dedup_average(wt[assay], ["compound_chembl_id", "target_accession"])
    mut_dedup[assay] = dedup_average(
        mut[assay].assign(mutation_sites_str=mut[assay]["mutation_sites"].apply(lambda s: ",".join(s))),
        ["compound_chembl_id", "target_accession", "mutation_sites_str"],
    )
    print(f"{assay}: WT 중복제거 후={len(wt_dedup[assay])}, 변이 중복제거 후={len(mut_dedup[assay])}")


## 6. affinity / interaction 계산

- `affinity = 9 - log10(standard_value)` (standard_value 단위는 nM, 즉 `-log10(value_M)`과 동일)
- `interaction = 0` if `standard_value == 10000`(ChEMBL이 비활성 측정값을 상한 1e4 nM로 캡핑하는 관례) else `1`

두 공식 모두 `DTA_dataset/chembl_35/IC50/fold1/train_data.csv`의 실제 값과 대조해서 확인했다
(예: CHEMBL138656/P22894, standard_value=2.0 → affinity=8.698970004336019, 정확히 일치).


In [ ]:
def add_affinity_interaction(df):
    df = df.copy()
    df["affinity"] = 9.0 - np.log10(df["standard_value"])
    df["interaction"] = (df["standard_value"] != 10000.0).astype(int)
    return df.drop(columns=["standard_value"])


for assay in ASSAYS:
    wt_dedup[assay] = add_affinity_interaction(wt_dedup[assay])
    mut_dedup[assay] = add_affinity_interaction(mut_dedup[assay])


## 7. target_accession에 변이 정보 합치고 WT + 변이 합치기

변이 데이터는 `target_accession(mut1,mut2,...)` 형태로 단백질 식별자를 다시 만든다
(예: `P00533(L858R,T790M)`). 이렇게 하면 WT와 변이가 서로 다른 "단백질"처럼 취급되어
PLM 임베딩을 독립적으로 뽑을 수 있다 (실제 `unique_fasta.csv`에 이 형식으로 저장돼 있음).

합칠 때 WT 쪽에는 `mutation_sites` 컬럼을 빈 문자열로 채워서 스키마를 맞춘다.


In [ ]:
combined = {}
for assay in ASSAYS:
    w = wt_dedup[assay].copy()
    w["mutation_sites"] = ""

    m = mut_dedup[assay].copy()
    m["mutation_sites"] = m["mutation_sites_str"]
    m["target_accession"] = m["target_accession"] + "(" + m["mutation_sites"] + ")"
    m = m.drop(columns=["mutation_sites_str"])

    cols = ["compound_chembl_id", "target_accession", "SMILES", "FASTA", "affinity", "interaction", "mutation_sites"]
    combined[assay] = pd.concat([w[cols], m[cols]], ignore_index=True)
    print(f"{assay}: WT {len(w)} + 변이 {len(m)} = 합계 {len(combined[assay])}")


## 8. 5-fold 분할

- 인덱스를 무작위로 섞은 뒤 5등분해서 각 조각을 fold별 test(20%)로 쓴다 (5개 조각을 합치면 전체 데이터와 같고,
  서로 겹치지 않는다 — 실제 파일로 검증한 성질과 일치).
- 남은 80%는 다시 무작위로 섞어 75:25로 나눠 train(60%) / valid(20%)로 쓴다.
- 정확한 random seed는 원본 노트북이 사라져서 알 수 없다. 여기서는 `SEED + fold_idx`를 사용한다
  (실행할 때마다 동일하게 재현되지만, 원본과 행 단위로 동일하지는 않을 수 있음).


In [ ]:
def make_folds(df, seed=SEED, n_splits=5):
    df = df.reset_index(drop=True)
    test_folds = kfold_indices(len(df), n_splits=n_splits, seed=seed)
    folds = []
    for fold_idx, test_idx in enumerate(test_folds):
        trainval_idx = np.setdiff1d(np.arange(len(df)), test_idx)
        train_idx, valid_idx = random_split(trainval_idx, test_size=0.25, seed=seed + fold_idx)
        folds.append({
            "train": df.iloc[train_idx],
            "valid": df.iloc[valid_idx],
            "test": df.iloc[test_idx],
        })
    return folds


all_folds = {assay: make_folds(combined[assay]) for assay in ASSAYS}
for assay in ASSAYS:
    f0 = all_folds[assay][0]
    print(f"{assay} fold1: train={len(f0['train'])}, valid={len(f0['valid'])}, test={len(f0['test'])}")


## 9. wild / mutant 서브셋

`mutation_sites`가 빈 문자열이면 WT, 아니면 변이로 나눈다.
(train/valid/test 각각에 대해 만들 수 있지만, 실제 저장소에는 `test_wild_data` / `test_mutant_data`처럼
test 스플릿에 대해서만 저장돼 있어 여기서도 test 기준으로만 만든다.)

`test_non5_*` 계열은 앞서 설명했듯 정확한 규칙을 복원하지 못해 **이 노트북에서는 만들지 않는다.**


In [ ]:
def split_wild_mutant(df):
    is_wild = df["mutation_sites"] == ""
    return df[is_wild].copy(), df[~is_wild].copy()


for assay in ASSAYS:
    for fold in all_folds[assay]:
        fold["test_wild"], fold["test_mutant"] = split_wild_mutant(fold["test"])


## 10. 최종 저장

`DTA_dataset/chembl_35/{IC50,Kd,Ki}/fold{1..5}/{train_data,valid_data,test_data,test_wild_data,test_mutant_data}.csv`
형태로 저장한다 (파일명은 `DTA_model/train.py`, `preprocess_dataset.py`가 그대로 기대하는 이름).

**주의**: 아래 셀은 기본적으로 주석 처리돼 있다. 이미 `DTA_dataset/chembl_35/`에 결과가 있으므로,
실수로 덮어쓰지 않도록 확인 후 직접 주석을 풀고 실행할 것.


In [ ]:
# for assay in ASSAYS:
#     for i, fold in enumerate(all_folds[assay], start=1):
#         out_dir = OUT_DIR / assay / f"fold{i}"
#         out_dir.mkdir(parents=True, exist_ok=True)
#         fold["train"].to_csv(out_dir / "train_data.csv", index=False)
#         fold["valid"].to_csv(out_dir / "valid_data.csv", index=False)
#         fold["test"].to_csv(out_dir / "test_data.csv", index=False)
#         fold["test_wild"].to_csv(out_dir / "test_wild_data.csv", index=False)
#         fold["test_mutant"].to_csv(out_dir / "test_mutant_data.csv", index=False)
#         print(f"saved {assay} fold{i}")
